# Insurance Charges — EDA and Modeling\nThis notebook contains data cleaning, EDA, modeling and interpretability steps. Run cells sequentially.

In [ ]:
import pandas as pd\nimport numpy as np\nfrom pathlib import Path\nfrom matplotlib import pyplot as plt\n\nDATA = Path('..') / 'insurance.csv'\ndf = pd.read_csv(DATA)\ndf.head()

## Cleaning & Feature engineering

In [ ]:
# Basic cleaning\ndf = df.copy()\n# No missing values in this dataset; show dtypes\ndf.info()

In [ ]:
# Feature engineering example\ndf['bmi_over_30'] = (df['bmi'] > 30).astype(int)\ndf['age_bucket'] = pd.cut(df['age'], bins=[17,25,35,45,55,65], labels=['18-25','26-35','36-45','46-55','56-65'])\ndf.head()

## Exploratory plots (saved to visuals/)

In [ ]:
from pathlib import Path\nPath('../visuals').mkdir(parents=True, exist_ok=True)\n# Load saved visuals if available\nfrom IPython.display import Image, display\nfor img in ['distribution_age.png','charges_vs_bmi.png','model_feature_importance.png']:\n    display(Image(filename=str(Path('..')/ 'visuals'/img)))

## Modeling (Linear Regression + Random Forest)

In [ ]:
from sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.linear_model import LinearRegression\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score\nimport joblib\n\nX = df.drop(columns=['charges'])\ny = df['charges']\ncat_cols = ['sex','smoker','region']\nnum_cols = ['age','bmi','children']\npreproc = ColumnTransformer([('num', StandardScaler(), num_cols), ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)])\nX_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)\n\nlr = Pipeline([('preproc', preproc), ('lr', LinearRegression())])\nrf = Pipeline([('preproc', preproc), ('rf', RandomForestRegressor(n_estimators=100, random_state=42))])\n\nlr.fit(X_train, y_train)\nrf.fit(X_train, y_train)\n\nfor name, model in [('LinearRegression', lr), ('RandomForest', rf)]:\n    preds = model.predict(X_test)\n    print(name)\n    print('MAE', mean_absolute_error(y_test, preds))\n    print('RMSE', mean_squared_error(y_test, preds, squared=False))\n    print('R2', r2_score(y_test, preds))\n\n# Save RF model\nPath('../models').mkdir(parents=True, exist_ok=True)\njoblib.dump(rf, '../models/rf_model.joblib')

## Interpretability (SHAP)

In [ ]:
# Try SHAP if available, otherwise use permutation importances\ntry:\n    import shap\n    explainer = shap.Explainer(rf.named_steps['rf'])\n    X_pre = rf.named_steps['preproc'].transform(X_test[:100])\n    shap_values = explainer(X_pre)\n    print('Computed SHAP values')\nexcept Exception as e:\n    print('SHAP not available or failed:', e)\n    from sklearn.inspection import permutation_importance\n    r = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42)\n    print('Top features by permutation importance:')\n    print(r.importances_mean)